# TaxGPT — Episode 10: Assembling the Full 131M-Parameter Model

Companion notebook to blog post *"Assembling a 131M-Parameter LLM From Scratch: The Full Architecture (Episode 10)"*.

Combines every component from Episodes 2-9 (embeddings, the transformer block, decoder-only stacking) into one complete model class, and verifies the parameter count directly rather than assuming it.

> **Verify before you publish:** this notebook uses TaxGPT's assumed spec (768-dim, 12 heads, 12 layers, 50,257 vocab) carried forward through the series. Confirm every number against your actual model definition.

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 4.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)

## 1. Every component, recapped from Episodes 6-8

In [2]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(emb_dim))
        self.beta = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta


class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, n_heads, context_len):
        super().__init__()
        assert emb_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads
        self.W_q = nn.Linear(emb_dim, emb_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, emb_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, emb_dim, bias=False)
        self.out_proj = nn.Linear(emb_dim, emb_dim)
        self.register_buffer("mask", torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        B, T, C = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class FeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden_dim)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, emb_dim)

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    """Decoder-only block (Episode 8 + 9): causal self-attention + FFN, no cross-attention."""
    def __init__(self, emb_dim, n_heads, hidden_dim, context_len):
        super().__init__()
        self.norm1 = LayerNorm(emb_dim)
        self.attn = MultiHeadAttention(emb_dim, n_heads, context_len)
        self.norm2 = LayerNorm(emb_dim)
        self.ff = FeedForward(emb_dim, hidden_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

print("LayerNorm, MultiHeadAttention, FeedForward, TransformerBlock defined (Episodes 6-8)")

LayerNorm, MultiHeadAttention, FeedForward, TransformerBlock defined (Episodes 6-8)


## 2. The full model: embeddings + N decoder blocks + output head

In [3]:
class TaxGPT(nn.Module):
    def __init__(self, vocab_size, emb_dim, n_heads, n_layers, hidden_dim, context_len, tie_weights=True):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.positional_embedding = nn.Embedding(context_len, emb_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(emb_dim, n_heads, hidden_dim, context_len)
            for _ in range(n_layers)
        ])

        self.final_norm = LayerNorm(emb_dim)
        self.output_head = nn.Linear(emb_dim, vocab_size, bias=False)

        if tie_weights:
            # output head reuses the token embedding table (transposed) -- saves ~38.6M params
            self.output_head.weight = self.token_embedding.weight

    def forward(self, token_ids):
        B, T = token_ids.shape
        tok_emb = self.token_embedding(token_ids)
        pos_emb = self.positional_embedding(torch.arange(T, device=token_ids.device))
        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        return logits

VOCAB_SIZE, EMB_DIM, N_HEADS, N_LAYERS, HIDDEN_DIM, CONTEXT_LEN = 50_257, 768, 12, 12, 3072, 1024

model_tied = TaxGPT(VOCAB_SIZE, EMB_DIM, N_HEADS, N_LAYERS, HIDDEN_DIM, CONTEXT_LEN, tie_weights=True)
model_untied = TaxGPT(VOCAB_SIZE, EMB_DIM, N_HEADS, N_LAYERS, HIDDEN_DIM, CONTEXT_LEN, tie_weights=False)

print("model instantiated -- both tied and untied variants, for direct comparison")

model instantiated -- both tied and untied variants, for direct comparison


## 3. Forward pass: confirming the shapes end to end

In [4]:
batch_size, seq_len = 2, 16
token_ids = torch.randint(0, VOCAB_SIZE, (batch_size, seq_len))

logits = model_tied(token_ids)
print("input token_ids shape:", token_ids.shape)
print("output logits shape:  ", logits.shape, " -> (batch, seq_len, vocab_size)")

# sanity check: logits should give a valid probability distribution per position after softmax
probs = F.softmax(logits[0, 0], dim=-1)
print("probs sum to 1.0 at position 0:", probs.sum().item())

input token_ids shape: torch.Size([2, 16])
output logits shape:   torch.Size([2, 16, 50257])  -> (batch, seq_len, vocab_size)
probs sum to 1.0 at position 0: 1.0


## 4. The real parameter count -- measured, not assumed

This is the number the blog post insists on checking directly rather than trusting a table.

In [5]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

total_tied = count_params(model_tied)
total_untied = count_params(model_untied)

print(f"TOTAL parameters, weight-tied:   {total_tied:,}")
print(f"TOTAL parameters, NOT tied:      {total_untied:,}")
print(f"difference (the tied embedding table): {total_untied - total_tied:,}")

TOTAL parameters, weight-tied:   124,412,160
TOTAL parameters, NOT tied:      163,009,536
difference (the tied embedding table): 38,597,376


## 5. Where the parameters actually live -- broken down by component

In [6]:
tok_emb_params = count_params(model_tied.token_embedding)
pos_emb_params = count_params(model_tied.positional_embedding)
blocks_params = count_params(model_tied.blocks)
final_norm_params = count_params(model_tied.final_norm)
# output head is 0 *additional* params when tied (reuses token_embedding's weight)
output_head_additional = 0 if model_tied.output_head.weight is model_tied.token_embedding.weight else count_params(model_tied.output_head)

print(f"{'Component':<28}{'Params':>15}{'% of total':>12}")
print("-" * 55)
for name, n in [
    ("Token embedding table", tok_emb_params),
    ("Positional embedding table", pos_emb_params),
    (f"{N_LAYERS} transformer blocks", blocks_params),
    ("Final LayerNorm", final_norm_params),
    ("Output head (additional)", output_head_additional),
]:
    print(f"{name:<28}{n:>15,}{n/total_tied:>11.1%}")
print("-" * 55)
print(f"{'TOTAL':<28}{total_tied:>15,}{1.0:>11.1%}")

Component                            Params  % of total
-------------------------------------------------------
Token embedding table            38,597,376      31.0%
Positional embedding table          786,432       0.6%
12 transformer blocks            85,026,816      68.3%
Final LayerNorm                       1,536       0.0%
Output head (additional)                  0       0.0%
-------------------------------------------------------
TOTAL                           124,412,160     100.0%


## 6. Confirming the decoder-only property: causality holds through the FULL model

Not just one attention layer -- the entire stacked model should never let an early token's output depend on a later token's input. Verified directly, not assumed.

In [7]:
model_tied.eval()
with torch.no_grad():
    ids_a = torch.randint(0, VOCAB_SIZE, (1, 10))
    ids_b = ids_a.clone()
    ids_b[0, 8] = torch.randint(0, VOCAB_SIZE, (1,))  # change a LATE token

    out_a = model_tied(ids_a)
    out_b = model_tied(ids_b)

    # positions BEFORE the changed token (0-7) should be identical -- they can't see position 8
    early_diff = (out_a[0, :8] - out_b[0, :8]).abs().max().item()
    # position 8 itself, and after, SHOULD differ
    late_diff = (out_a[0, 8:] - out_b[0, 8:]).abs().max().item()

print(f"max difference at positions 0-7 (should be ~0.0): {early_diff:.2e}")
print(f"max difference at positions 8-9 (should be > 0):  {late_diff:.4f}")
print()
print("PASS: early positions are unaffected by a change to a later token -- causality confirmed end to end"
      if early_diff < 1e-5 else "WARNING: causality violated somewhere in the stack")

max difference at positions 0-7 (should be ~0.0): 0.00e+00
max difference at positions 8-9 (should be > 0):  443.5207

PASS: early positions are unaffected by a change to a later token -- causality confirmed end to end


## Takeaway

Every episode in this series was building toward a short, unremarkable `forward()` method: embed, run through N decoder blocks, project to vocabulary. The real work was making sure each piece (attention, LayerNorm, residuals, the feed-forward block, decoder-only causal masking) was individually correct -- verified here end to end with a real parameter count and a real causality check, not just assumed from the architecture diagram.

**Next notebook: Episode 11 — The training loop.**